# NYISO Solar Forecasting: Data Acquisition and Integration

## Purpose

## Research Context

## Inputs and Outputs

## Imports and Configuration

In [1]:
import os
import zipfile

from pathlib import Path
from loguru import logger

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [3]:
from solar_forecast.config import (
    PROCESSED_ROOT,
    SOLAR_RAW_ROOT,
    SOLAR_ZIP_PATH,
    UNZIPPED_ROOTS,
    NYISO_OUT,
    ERA5_OUT,
    MERGED_OUT,
    TS_COL   as ts_col,
    ZONE_COL as zone_col,
)

from solar_forecast.dataset import (
    unzip_main_archive,
    unzip_all_archives,
    load_folder,
    ensure_required_columns,
    resolve_value_col,
    parse_nyiso_time,
)

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

## Data Extraction 

### Raw Archive Extraction

In [4]:
unzip_main_archive(SOLAR_ZIP_PATH, SOLAR_RAW_ROOT)

unzip_all_archives(SOLAR_RAW_ROOT / "actuals",   UNZIPPED_ROOTS["actuals"])
unzip_all_archives(SOLAR_RAW_ROOT / "forecasts", UNZIPPED_ROOTS["forecasts"])
unzip_all_archives(SOLAR_RAW_ROOT / "capacity",  UNZIPPED_ROOTS["capacity"])

2026-03-28 16:28:50.071 | INFO     | solar_forecast.dataset:unzip_main_archive:25 - extracted main: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar.zip
2026-03-28 16:28:50.071 | WARNING  | solar_forecast.dataset:unzip_all_archives:38 - input folder not found: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar/actuals
2026-03-28 16:28:50.072 | WARNING  | solar_forecast.dataset:unzip_all_archives:38 - input folder not found: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar/forecasts
2026-03-28 16:28:50.072 | WARNING  | solar_forecast.dataset:unzip_all_archives:38 - input folder not found: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar/capacity


## Load CSVs Into DataFrames

In [5]:
df_actual   = load_folder(UNZIPPED_ROOTS["actuals"])
df_forecast = load_folder(UNZIPPED_ROOTS["forecasts"])
df_capacity = load_folder(UNZIPPED_ROOTS["capacity"])

print("Loaded Shapes")
print(". . .")
print("Actuals:",   df_actual.shape)
print("Forecasts:", df_forecast.shape)
print("Capacity:",  df_capacity.shape)

Loaded Shapes
. . .
Actuals: (500916, 5)
Forecasts: (506004, 5)
Capacity: (10716, 5)


In [6]:
for df in (df_actual, df_forecast, df_capacity):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

ensure_required_columns(df_actual,   "df_actual")
ensure_required_columns(df_forecast, "df_forecast")
ensure_required_columns(df_capacity, "df_capacity")

In [7]:
actual_val_col   = resolve_value_col(df_actual)
forecast_val_col = resolve_value_col(df_forecast)
capacity_val_col = resolve_value_col(df_capacity)

print("Resolved Value Columns")
print(". . .")
print("Actuals:   ", actual_val_col)
print("Forecasts: ", forecast_val_col)
print("Capacity:  ", capacity_val_col)

Resolved Value Columns
. . .
Actuals:    mw_value
Forecasts:  mw_value
Capacity:   mw_value


In [8]:
df_actual[actual_val_col]     = pd.to_numeric(df_actual[actual_val_col],     errors="coerce")
df_forecast[forecast_val_col] = pd.to_numeric(df_forecast[forecast_val_col], errors="coerce")
df_capacity[capacity_val_col] = pd.to_numeric(df_capacity[capacity_val_col], errors="coerce")

df_actual   = df_actual.rename(columns={actual_val_col:   "actual_mw"})
df_forecast = df_forecast.rename(columns={forecast_val_col: "forecast_mw"})
df_capacity = df_capacity.rename(columns={capacity_val_col: "capacity_mw"})

print("\nColumns After Canonical Renaming")
print(". . .")
print("Actual:",   df_actual.columns.tolist())
print("Forecast:", df_forecast.columns.tolist())
print("Capacity:", df_capacity.columns.tolist())


Columns After Canonical Renaming
. . .
Actual: ['time_stamp', 'time_zone', 'zone_name', 'actual_mw', 'source_file']
Forecast: ['time_stamp', 'time_zone', 'zone_name', 'forecast_mw', 'source_file']
Capacity: ['time_stamp', 'time_zone', 'zone_name', 'capacity_mw', 'source_file']


In [9]:
df_actual_hourly = (
    df_actual
    .dropna(subset=[ts_col, zone_col, "actual_mw"])
    .groupby([ts_col, zone_col], as_index=False)["actual_mw"]
    .sum()
)

df_forecast_hourly = (
    df_forecast
    .dropna(subset=[ts_col, zone_col, "forecast_mw"])
    .groupby([ts_col, zone_col], as_index=False)["forecast_mw"]
    .sum()
)

df_capacity_updates = (
    df_capacity
    .dropna(subset=[ts_col, zone_col, "capacity_mw"])
    .sort_values([zone_col, ts_col, "source_file"])
    .groupby([ts_col, zone_col], as_index=False)["capacity_mw"]
    .last()
)

print("\nPost-Aggregation Shapes")
print(". . .")
print("Actual Hourly:",    df_actual_hourly.shape)
print("Forecast Hourly:",  df_forecast_hourly.shape)
print("Capacity Updated:", df_capacity_updates.shape)


Post-Aggregation Shapes
. . .
Actual Hourly: (500868, 3)
Forecast Hourly: (505956, 3)
Capacity Updated: (10716, 3)


In [10]:
df_nyiso = (
    df_actual_hourly
    .merge(df_forecast_hourly, how="outer", on=[ts_col, zone_col])
    .sort_values([zone_col, ts_col])
    .reset_index(drop=True)
)

df_nyiso = (
    df_nyiso
    .merge(df_capacity_updates, how="left", on=[ts_col, zone_col])
    .sort_values([zone_col, ts_col])
    .reset_index(drop=True)
)

df_nyiso["capacity_mw"] = (
    df_nyiso
    .groupby(zone_col)["capacity_mw"]
    .ffill()
)

df_nyiso.to_csv(NYISO_OUT, index=False)
print("Saved NYISO Dataset:", NYISO_OUT)

Saved NYISO Dataset: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/01_nyiso_merged.csv


In [14]:
df_era5 = pd.read_csv(ERA5_OUT, low_memory=False)

df_era5.columns = (
    df_era5.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

if "time_stamp" in df_era5.columns:
    df_era5[ts_col] = pd.to_datetime(df_era5["time_stamp"], utc=True, errors="coerce")
elif "time" in df_era5.columns:
    df_era5[ts_col] = pd.to_datetime(df_era5["time"], utc=True, errors="coerce")
else:
    raise KeyError(f"ERA5 time not found. The columns were: {df_era5.columns.tolist()}")

df_era5[ts_col] = df_era5[ts_col].dt.floor("h")
df_era5 = df_era5.dropna(subset=[ts_col])
df_era5 = df_era5.groupby(ts_col, as_index=False).first()

df_era5.to_csv(ERA5_OUT, index=False)
print("Saved ERA5 Dataset:", ERA5_OUT)

Saved ERA5 Dataset: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/02_era5_weather.csv


In [17]:
df_nyiso[ts_col] = pd.to_datetime(df_nyiso[ts_col], utc=True, errors="coerce")  #add — reparse after csv write strips tz dtype

df_merge = pd.merge(df_nyiso, df_era5, on=ts_col, how="inner")
df_merge = df_merge.sort_values([ts_col, zone_col]).reset_index(drop=True)

df_merge.to_csv(MERGED_OUT, index=False)

print("NYISO Shape:",  df_nyiso.shape)
print("ERA5 Shape:",   df_era5.shape)
print("Merged Shape:", df_merge.shape)

print("Merged Time Range:", df_merge[ts_col].min(), "→", df_merge[ts_col].max())
print("Saved Merged Dataset:", MERGED_OUT)

NYISO Shape: (509412, 5)
ERA5 Shape: (51000, 8)
Merged Shape: (509412, 12)
Merged Time Range: 2020-11-17 00:00:00+00:00 → 2025-09-20 23:00:00+00:00
Saved Merged Dataset: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/03_merged_data.csv


In [18]:
print("Columns Check")
print(". . .")
print("NYISO Columns:",  df_nyiso.columns.tolist())
print("ERA5 Columns:",   df_era5.columns.tolist())
print("Merged Columns:", df_merge.columns.tolist())

print("Timestamp Missing Check")
print(". . .")
print("NYISO Missing time_stamp:",  df_nyiso[ts_col].isna().sum())
print("ERA5 Missing time_stamp:",   df_era5[ts_col].isna().sum())
print("Merged Missing time_stamp:", df_merge[ts_col].isna().sum())

merge_validation = pd.DataFrame(
    {
        "dataset": ["nyiso", "era5", "merged"],
        "rows":    [len(df_nyiso), len(df_era5), len(df_merge)],
        "time_min": [df_nyiso[ts_col].min(), df_era5[ts_col].min(), df_merge[ts_col].min()],
        "time_max": [df_nyiso[ts_col].max(), df_era5[ts_col].max(), df_merge[ts_col].max()],
        "missing_time_stamp": [
            df_nyiso[ts_col].isna().sum(),
            df_era5[ts_col].isna().sum(),
            df_merge[ts_col].isna().sum(),
        ],
    }
)

merge_validation

Columns Check
. . .
NYISO Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw']
ERA5 Columns: ['time_stamp', 'time', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']
Merged Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'time', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']
Timestamp Missing Check
. . .
NYISO Missing time_stamp: 0
ERA5 Missing time_stamp: 0
Merged Missing time_stamp: 0


,dataset,rows,time_min,time_max,missing_time_stamp
0,nyiso,509412,2020-11-17 00:00:00+00:00,2025-09-20 23:00:00+00:00,0
1,era5,51000,2020-01-01 00:00:00+00:00,2025-10-25 23:00:00+00:00,0
2,merged,509412,2020-11-17 00:00:00+00:00,2025-09-20 23:00:00+00:00,0


## Summary
In this notebook